In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import train_test_split

from torch.utils.data import (
    TensorDataset,
    DataLoader
)

In [2]:
encoded_sequences = np.load(
    "../encoded_sequences_len40_v2.npy"
)

print(
    encoded_sequences.shape
)

(7405, 42)


In [3]:
X_train, X_test = train_test_split(
    encoded_sequences,
    test_size=0.1,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(6664, 42)
(741, 42)


In [4]:
X_train = torch.tensor(
    X_train,
    dtype=torch.long
)

X_test = torch.tensor(
    X_test,
    dtype=torch.long
)

In [5]:
train_dataset = TensorDataset(
    X_train
)

test_dataset = TensorDataset(
    X_test
)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False
)

In [6]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [7]:
class PositionalEncoding(
    nn.Module
):

    def __init__(
        self,
        d_model,
        max_len=42
    ):

        super().__init__()

        pe = torch.zeros(
            max_len,
            d_model
        )

        position = torch.arange(
            0,
            max_len
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2
            )
            *
            (
                -torch.log(
                    torch.tensor(
                        10000.0
                    )
                )
                /
                d_model
            )
        )

        pe[:,0::2] = torch.sin(
            position *
            div_term
        )

        pe[:,1::2] = torch.cos(
            position *
            div_term
        )

        pe = pe.unsqueeze(0)

        self.register_buffer(
            "pe",
            pe
        )

    def forward(
        self,
        x
    ):

        return (
            x
            +
            self.pe[:,:x.size(1)]
        )

In [8]:
class TransformerVAE(
    nn.Module
):

    def __init__(
        self,
        vocab_size=23,
        embed_dim=128,
        latent_dim=64,
        max_len=42
    ):

        super().__init__()

        self.max_len = max_len

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=0
        )

        self.pos_encoder = (
            PositionalEncoding(
                embed_dim,
                max_len
            )
        )

        encoder_layer = (
            nn.TransformerEncoderLayer(
                d_model=embed_dim,
                nhead=8,
                batch_first=True
            )
        )

        self.encoder = (
            nn.TransformerEncoder(
                encoder_layer,
                num_layers=3
            )
        )

        self.fc_mu = nn.Linear(
            embed_dim,
            latent_dim
        )

        self.fc_logvar = nn.Linear(
            embed_dim,
            latent_dim
        )

        self.latent_to_embed = (
            nn.Linear(
                latent_dim,
                embed_dim
            )
        )

        decoder_layer = (
            nn.TransformerDecoderLayer(
                d_model=embed_dim,
                nhead=8,
                batch_first=True
            )
        )

        self.decoder = (
            nn.TransformerDecoder(
                decoder_layer,
                num_layers=3
            )
        )

        self.output_layer = nn.Linear(
            embed_dim,
            vocab_size
        )

    def encode(
        self,
        x
    ):

        x = self.embedding(x)

        x = self.pos_encoder(x)

        enc = self.encoder(x)

        pooled = enc.mean(dim=1)

        mu = self.fc_mu(
            pooled
        )

        logvar = self.fc_logvar(
            pooled
        )

        return mu, logvar
        
    def reparameterize(
        self,
        mu,
        logvar
    ):

        std = torch.exp(
            0.5 * logvar
        )

        eps = torch.randn_like(
            std
        )

        return (
            mu
            +
            eps * std
        )

    def decode(
        self,
        z,
        decoder_input
    ):

        tgt = self.embedding(
            decoder_input
        )

        tgt = self.pos_encoder(
            tgt
        )

        memory = (
            self.latent_to_embed(
                z
            )
            .unsqueeze(1)
        )

        out = self.decoder(
            tgt,
            memory
        )

        logits = self.output_layer(
            out
        )

        return logits

    def forward(
        self,
        x
    ):

        mu, logvar = self.encode(
            x
        )

        z = self.reparameterize(
            mu,
            logvar
        )

        decoder_input = x[:,:-1]

        logits = self.decode(
            z,
            decoder_input
        )

        return (
            logits,
            mu,
            logvar
        )

In [9]:
vae = TransformerVAE(
    vocab_size=23,
    max_len=42
).to(device)

print(
    sum(
        p.numel()
        for p in vae.parameters()
    )
)

3787799


In [10]:
def vae_loss(
    logits,
    target,
    mu,
    logvar
):

    target = target[:,1:]

    recon_loss = F.cross_entropy(
        logits.reshape(-1,23),
        target.reshape(-1),
        ignore_index=0
    )

    kl_loss = -0.5 * torch.mean(
        1
        +
        logvar
        -
        mu.pow(2)
        -
        logvar.exp()
    )

    loss = (
        recon_loss
        +
        0.01 * kl_loss
    )

    return (
        loss,
        recon_loss,
        kl_loss
    )

In [11]:
optimizer = torch.optim.Adam(
    vae.parameters(),
    lr=1e-4
)

In [12]:
print(
    sum(
        p.numel()
        for p in vae.parameters()
    )
)

3787799


In [13]:
num_epochs = 20
for epoch in range(num_epochs):

    vae.train()

    total_loss = 0
    total_recon = 0
    total_kl = 0

    for batch in train_loader:

        x = batch[0].to(device)

        logits, mu, logvar = vae(x)

        loss, recon, kl = vae_loss(
            logits,
            x,
            mu,
            logvar
        )

        optimizer.zero_grad()

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            vae.parameters(),
            1.0
        )

        optimizer.step()

        total_loss += loss.item()
        total_recon += recon.item()
        total_kl += kl.item()

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Loss={total_loss/len(train_loader):.4f} "
        f"Recon={total_recon/len(train_loader):.4f} "
        f"KL={total_kl/len(train_loader):.4f}"
    )

Epoch [1/20] Loss=2.7830 Recon=2.7809 KL=0.2081
Epoch [2/20] Loss=2.4921 Recon=2.4893 KL=0.2785
Epoch [3/20] Loss=2.3089 Recon=2.3044 KL=0.4436
Epoch [4/20] Loss=2.1413 Recon=2.1351 KL=0.6116
Epoch [5/20] Loss=1.8489 Recon=1.8411 KL=0.7820
Epoch [6/20] Loss=1.2543 Recon=1.2447 KL=0.9593
Epoch [7/20] Loss=0.7808 Recon=0.7698 KL=1.0982
Epoch [8/20] Loss=0.5408 Recon=0.5291 KL=1.1716
Epoch [9/20] Loss=0.3989 Recon=0.3869 KL=1.1991
Epoch [10/20] Loss=0.3098 Recon=0.2979 KL=1.1944
Epoch [11/20] Loss=0.2363 Recon=0.2246 KL=1.1732
Epoch [12/20] Loss=0.1820 Recon=0.1706 KL=1.1373
Epoch [13/20] Loss=0.1299 Recon=0.1189 KL=1.0952
Epoch [14/20] Loss=0.0970 Recon=0.0865 KL=1.0443
Epoch [15/20] Loss=0.0732 Recon=0.0633 KL=0.9861
Epoch [16/20] Loss=0.0626 Recon=0.0534 KL=0.9135
Epoch [17/20] Loss=0.0509 Recon=0.0425 KL=0.8444
Epoch [18/20] Loss=0.0440 Recon=0.0364 KL=0.7669
Epoch [19/20] Loss=0.0382 Recon=0.0313 KL=0.6888
Epoch [20/20] Loss=0.0341 Recon=0.0280 KL=0.6153


In [14]:
vae.eval()

correct = 0
total = 0

with torch.no_grad():

    for batch in test_loader:

        x = batch[0].to(device)

        logits, _, _ = vae(x)

        pred = logits.argmax(dim=2)

        target = x[:,1:]

        mask = (target != 0)

        correct += (
            ((pred == target) & mask)
            .sum()
            .item()
        )

        total += (
            mask
            .sum()
            .item()
        )

token_acc = correct / total

print(
    f"Token Accuracy: {token_acc:.4f}"
)

Token Accuracy: 0.9999


In [15]:
vae.eval()

with torch.no_grad():

    x = X_test[:1].to(device)

    logits, _, _ = vae(x)

    pred = logits.argmax(dim=2)

print("TARGET")
print(
    x[0][:25].cpu().numpy()
)

print("PRED")
print(
    pred[0][:25].cpu().numpy()
)

TARGET
[ 1 19 12 12  4  5  4 15 14  8 15 21 20 21 20 15  3  7  4 16  3  2  0  0
  0]
PRED
[19 12 12  4  5  4 15 14  8 15 21 20 21 20 15  3  7  4 16  3  2  2  2  2
  2]


In [16]:
torch.save(
    vae.state_dict(),
    "transformer_vae_len40.pt"
)